# Human vs Model Performance Analysis on Blind VQA

This notebook demonstrates comprehensive analysis methods for comparing human and model performance on blind VQA datasets.

## Dataset Mapping
- **Choice questions**: MMStar dataset
- **Text questions**: VQA1K dataset

## Analysis Methods
1. Overall agreement rates
2. Performance by answer type (choice vs text)
3. Performance by question category  
4. Confidence calibration
5. Error analysis
6. Confusion matrices

In [ ]:
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter, defaultdict
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score

# Import the analyzer
sys.path.append('..')
from compare_human_model_pipeline import HumanModelAnalyzer

# Plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Setup complete!")

## 1. Setup Paths

Define paths to your data files based on answer type.

In [ ]:
# Dataset mapping: answer_type -> dataset name
DATASET_MAPPING = {
    'choice': 'mmstar',
    'text': 'vqa1k'
}

# Choose which analysis to run
ANSWER_TYPE = 'choice'  # or 'text'
DATASET = DATASET_MAPPING[ANSWER_TYPE]

# Paths
BASE_DIR = Path('/home/user/HPA')

# Human results
if ANSWER_TYPE == 'choice':
    HUMAN_RESULTS = BASE_DIR / 'data/training/s1_choice/cleaned_n14_choice.json'
    QUESTIONS = BASE_DIR / 'dataset/questions/s1.csv'
else:
    HUMAN_RESULTS = BASE_DIR / 'data/training/s1_text/cleaned_n14_text.json'
    QUESTIONS = BASE_DIR / 'dataset/questions/s1.csv'

# Model results (adjust to your model output path)
MODEL_RESULTS = BASE_DIR / f'outputs/results/model_{DATASET}_blind.jsonl'

# Output directory
OUTPUT_DIR = BASE_DIR / f'analysis/results_{ANSWER_TYPE}_{DATASET}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Analysis configuration:")
print(f"  Answer type: {ANSWER_TYPE}")
print(f"  Dataset: {DATASET}")
print(f"  Human results: {HUMAN_RESULTS}")
print(f"  Model results: {MODEL_RESULTS}")
print(f"  Output: {OUTPUT_DIR}")

## 2. Load and Explore Data

Load human responses, model predictions, and question metadata.

In [ ]:
# Initialize analyzer
analyzer = HumanModelAnalyzer(output_dir=str(OUTPUT_DIR))

# Load data
human_df = analyzer.load_human_results(str(HUMAN_RESULTS))
model_df = analyzer.load_model_results(str(MODEL_RESULTS))
questions_df = analyzer.load_questions(str(QUESTIONS))

print(f"\nData shapes:")
print(f"  Human: {human_df.shape}")
print(f"  Model: {model_df.shape}")
print(f"  Questions: {questions_df.shape}")

In [ ]:
# Explore human data
print("=== Human Data Sample ===")
print(human_df.head())

print("\n=== Answer Type Distribution ===")
if 'answer_type' in human_df.columns:
    print(human_df['answer_type'].value_counts())

print("\n=== Questions per participant ===")
if 'participant_id' in human_df.columns:
    print(human_df.groupby('participant_id')['qid'].nunique().describe())

In [ ]:
# Explore model data
print("=== Model Data Sample ===")
print(model_df.head())

print("\n=== Model Answer Distribution (first 10) ===")
if 'model_answer_normalized' in model_df.columns:
    print(model_df['model_answer_normalized'].value_counts().head(10))

## 3. Filter by Answer Type

Separate choice questions (MMStar) from text questions (VQA1K).

In [ ]:
# Merge questions with answer_type info
if 'answer_type' in questions_df.columns:
    human_df = pd.merge(
        human_df,
        questions_df[['qid', 'answer_type']],
        on='qid',
        how='left',
        suffixes=('', '_q')
    )

    # Filter by answer type
    human_df_filtered = human_df[human_df['answer_type'] == ANSWER_TYPE]
    
    print(f"Filtered to {ANSWER_TYPE} questions:")
    print(f"  Before: {len(human_df)} responses")
    print(f"  After: {len(human_df_filtered)} responses")
    print(f"  Unique questions: {human_df_filtered['qid'].nunique()}")
    
    human_df = human_df_filtered
else:
    print("Warning: No answer_type column in questions data")

## 4. Aggregate Human Responses

Combine multiple human responses per question to get consensus.

In [ ]:
# Aggregate
human_agg = analyzer.aggregate_human_responses(human_df)

print("=== Aggregated Human Data ===")
print(human_agg.head())

print("\n=== Consensus Rate Distribution ===")
plt.figure(figsize=(10, 5))
plt.hist(human_agg['human_consensus_rate'], bins=20, edgecolor='black')
plt.xlabel('Consensus Rate (% agreeing with majority)')
plt.ylabel('Number of Questions')
plt.title('Distribution of Human Consensus Rates')
plt.axvline(human_agg['human_consensus_rate'].mean(), color='r', linestyle='--', 
            label=f'Mean: {human_agg["human_consensus_rate"].mean():.2f}')
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'human_consensus_distribution.png', dpi=300)
plt.show()

## 5. Merge Human and Model Data

Match by qid to enable direct comparison.

In [ ]:
# Merge
merged_df = analyzer.merge_human_model(human_agg, model_df, questions_df)

# Compute agreement
merged_df = analyzer.compute_agreement(merged_df)

print(f"=== Merged Dataset ===")
print(f"  Total questions: {len(merged_df)}")
print(f"  Columns: {list(merged_df.columns)}")
print("\n=== Sample ===")
print(merged_df[['qid', 'human_answer', 'model_answer_normalized', 'exact_match', 
                 'human_consensus_rate', 'category']].head(10))

## 6. Overall Performance Analysis

Compute overall agreement and accuracy metrics.

In [ ]:
overall_metrics = analyzer.analyze_overall_performance(merged_df)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Agreement pie chart
agree_counts = merged_df['exact_match'].value_counts()
axes[0].pie(agree_counts, labels=['Disagree', 'Agree'], autopct='%1.1f%%', 
            colors=['#ff9999', '#99ff99'], startangle=90)
axes[0].set_title(f'Human-Model Agreement\n({ANSWER_TYPE} questions)')

# Bar chart of metrics
metrics_to_plot = {
    'Agreement Rate': overall_metrics['agreement_rate'],
    'Human Consensus': overall_metrics['human_avg_consensus']
}
if overall_metrics['human_avg_confidence']:
    metrics_to_plot['Human Confidence\n(scaled to 0-1)'] = overall_metrics['human_avg_confidence'] / 5

axes[1].bar(metrics_to_plot.keys(), metrics_to_plot.values(), color=['steelblue', 'coral', 'lightgreen'])
axes[1].set_ylim([0, 1])
axes[1].set_ylabel('Rate')
axes[1].set_title('Performance Metrics')
axes[1].axhline(y=0.5, color='r', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'overall_performance.png', dpi=300)
plt.show()

## 7. Performance by Category

Analyze which question categories show high/low agreement.

In [ ]:
by_category = analyzer.analyze_by_category(merged_df)

if by_category is not None and len(by_category) > 0:
    # Plot top/bottom categories
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Top 10 categories (best agreement)
    top10 = by_category.nlargest(10, 'agreement_rate')
    axes[0].barh(range(len(top10)), top10['agreement_rate'], color='green', alpha=0.7)
    axes[0].set_yticks(range(len(top10)))
    axes[0].set_yticklabels(top10['category'])
    axes[0].set_xlabel('Agreement Rate')
    axes[0].set_title('Top 10 Categories (Highest Agreement)')
    axes[0].set_xlim([0, 1])
    
    # Bottom 10 categories (lowest agreement)
    bottom10 = by_category.nsmallest(10, 'agreement_rate')
    axes[1].barh(range(len(bottom10)), bottom10['agreement_rate'], color='red', alpha=0.7)
    axes[1].set_yticks(range(len(bottom10)))
    axes[1].set_yticklabels(bottom10['category'])
    axes[1].set_xlabel('Agreement Rate')
    axes[1].set_title('Bottom 10 Categories (Lowest Agreement)')
    axes[1].set_xlim([0, 1])
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'category_performance.png', dpi=300)
    plt.show()

## 8. Confidence Calibration Analysis

Examine relationship between human confidence and agreement with model.

In [ ]:
if 'human_confidence' in merged_df.columns:
    # Bin by confidence
    merged_df['conf_bin'] = pd.cut(merged_df['human_confidence'], 
                                     bins=[0, 1, 2, 3, 4, 5], 
                                     labels=['1', '2', '3', '4', '5'])
    
    # Compute metrics per bin
    conf_analysis = merged_df.groupby('conf_bin').agg({
        'exact_match': ['mean', 'count'],
        'human_consensus_rate': 'mean'
    }).reset_index()
    
    print("=== Agreement by Confidence ===")
    print(conf_analysis)
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    
    x = range(len(conf_analysis))
    width = 0.35
    
    ax.bar([i - width/2 for i in x], conf_analysis['exact_match']['mean'], 
           width, label='Agreement Rate', color='steelblue')
    ax.bar([i + width/2 for i in x], conf_analysis['human_consensus_rate']['mean'], 
           width, label='Human Consensus', color='coral')
    
    ax.set_xlabel('Human Confidence Level')
    ax.set_ylabel('Rate')
    ax.set_title('Agreement and Consensus by Human Confidence')
    ax.set_xticks(x)
    ax.set_xticklabels(conf_analysis['conf_bin'])
    ax.legend()
    ax.set_ylim([0, 1])
    
    # Add counts
    for i, count in enumerate(conf_analysis['exact_match']['count']):
        ax.text(i, 0.05, f'n={int(count)}', ha='center', fontsize=9)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'confidence_calibration.png', dpi=300)
    plt.show()

## 9. Disagreement Analysis

Deep dive into cases where human and model disagree.

In [ ]:
disagreements = analyzer.analyze_disagreements(merged_df)

print("\n=== Sample Disagreements ===")
sample_cols = ['qid', 'question', 'human_answer', 'model_answer_normalized', 
               'human_consensus_rate', 'category']
available_cols = [col for col in sample_cols if col in disagreements.columns]
print(disagreements[available_cols].head(10).to_string())

# Analyze common error patterns for choice questions
if ANSWER_TYPE == 'choice':
    print("\n=== Error Pattern Analysis ===")
    error_pairs = disagreements.groupby(['human_answer', 'model_answer_normalized']).size()
    print("\nTop 10 Human→Model error pairs:")
    print(error_pairs.nlargest(10))

## 10. Confusion Matrix (Choice Questions Only)

Visual representation of human vs model answer patterns.

In [ ]:
if ANSWER_TYPE == 'choice':
    analyzer.plot_confusion_matrix(merged_df, answer_type='choice')
    
    # Also show as percentage
    all_answers = sorted(set(merged_df['human_answer'].unique()) | 
                        set(merged_df['model_answer_normalized'].unique()))
    
    cm = confusion_matrix(merged_df['human_answer'], 
                         merged_df['model_answer_normalized'], 
                         labels=all_answers)
    
    # Normalize by row (human answer)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='RdYlGn', 
                xticklabels=all_answers, yticklabels=all_answers, ax=ax)
    ax.set_xlabel('Model Answer')
    ax.set_ylabel('Human Answer (Consensus)')
    ax.set_title('Confusion Matrix (Normalized by Human Answer)')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'confusion_matrix_normalized.png', dpi=300)
    plt.show()

## 11. Answer Distribution Comparison

Compare the distribution of answers between humans and model.

In [ ]:
analyzer.plot_answer_distribution(merged_df)

# Statistical comparison
if ANSWER_TYPE == 'choice':
    print("=== Answer Distribution Statistics ===")
    
    human_dist = merged_df['human_answer'].value_counts(normalize=True).sort_index()
    model_dist = merged_df['model_answer_normalized'].value_counts(normalize=True).sort_index()
    
    comparison = pd.DataFrame({
        'Human %': human_dist * 100,
        'Model %': model_dist * 100
    })
    comparison['Difference'] = comparison['Model %'] - comparison['Human %']
    
    print(comparison)
    
    # Chi-square test
    from scipy.stats import chisquare
    
    # Align distributions
    all_answers = sorted(set(human_dist.index) | set(model_dist.index))
    human_counts = [merged_df[merged_df['human_answer'] == a].shape[0] for a in all_answers]
    model_counts = [merged_df[merged_df['model_answer_normalized'] == a].shape[0] for a in all_answers]
    
    chi2, p_value = chisquare(model_counts, human_counts)
    print(f"\nChi-square test:")
    print(f"  Chi2 statistic: {chi2:.4f}")
    print(f"  p-value: {p_value:.4f}")
    print(f"  Distributions {'significantly different' if p_value < 0.05 else 'not significantly different'} (α=0.05)")

## 12. Save Results

Export all analysis results for further use.

In [ ]:
# Save merged data
merged_df.to_csv(OUTPUT_DIR / 'merged_analysis.csv', index=False)
print(f"Saved: {OUTPUT_DIR / 'merged_analysis.csv'}")

# Save disagreements
disagreements.to_csv(OUTPUT_DIR / 'disagreements.csv', index=False)
print(f"Saved: {OUTPUT_DIR / 'disagreements.csv'}")

# Save metrics
all_metrics = {
    'overall': overall_metrics,
    'by_category': by_category.to_dict('records') if by_category is not None else None,
}

with open(OUTPUT_DIR / 'metrics.json', 'w') as f:
    json.dump(all_metrics, f, indent=2)
print(f"Saved: {OUTPUT_DIR / 'metrics.json'}")

print(f"\nAll results saved to: {OUTPUT_DIR}")

## 13. Summary Report

Generate a concise summary of findings.

In [ ]:
print("="*60)
print(f"ANALYSIS SUMMARY - {ANSWER_TYPE.upper()} QUESTIONS ({DATASET})")
print("="*60)

print(f"\n📊 Dataset:")
print(f"  Total questions analyzed: {len(merged_df)}")
print(f"  Average responses per question: {overall_metrics['total_questions'] / merged_df['qid'].nunique():.1f}")

print(f"\n👥 Human Performance:")
print(f"  Average consensus rate: {overall_metrics['human_avg_consensus']:.1%}")
if overall_metrics['human_avg_confidence']:
    print(f"  Average confidence: {overall_metrics['human_avg_confidence']:.2f}/5.0")

print(f"\n🤖 Human-Model Agreement:")
print(f"  Overall agreement rate: {overall_metrics['agreement_rate']:.1%}")
print(f"  Total agreements: {merged_df['exact_match'].sum()}/{len(merged_df)}")
print(f"  Total disagreements: {len(disagreements)}/{len(merged_df)}")

if by_category is not None and len(by_category) > 0:
    print(f"\n🎯 Best Category: {by_category.iloc[0]['category']}")
    print(f"     Agreement: {by_category.iloc[0]['agreement_rate']:.1%}")
    print(f"\n⚠️  Worst Category: {by_category.iloc[-1]['category']}")
    print(f"     Agreement: {by_category.iloc[-1]['agreement_rate']:.1%}")

print("\n" + "="*60)
print("Analysis complete! Check plots and CSV files for details.")
print("="*60)